In [ ]:
from hcc_release_paths import project_path as _hcc_path
# =============================================================================
# Step27 Cell 1
# Initialization + Frozen Input Verification
# =============================================================================

from pathlib import Path
import sys
import os
import json
import platform
import random
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import scipy
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# -----------------------------------------------------------------------------
# 0. Reproducibility
# -----------------------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

# -----------------------------------------------------------------------------
# 1. Fixed project paths
# -----------------------------------------------------------------------------

STEP27_DIR = Path(_hcc_path('LIHC_scRNA_Step27_Trajectory'))

INPUT_H5AD = Path(
    r"E:\TCGA\LIHC_scRNA_Step26_MS3_Biology\Data"
    r"\GSE149614_STEP26_FINAL_FROZEN_MS3_BIOLOGY.h5ad"
)

TABLE_DIR = STEP27_DIR / "Tables"
FIGURE_DIR = STEP27_DIR / "Figures"
DATA_DIR = STEP27_DIR / "Data"
AUDIT_DIR = STEP27_DIR / "Audit"
MANIFEST_DIR = STEP27_DIR / "Manifest"

for d in [
    STEP27_DIR,
    TABLE_DIR,
    FIGURE_DIR,
    DATA_DIR,
    AUDIT_DIR,
    MANIFEST_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

sc.settings.figdir = str(FIGURE_DIR)

# Figure defaults
plt.rcParams["savefig.dpi"] = 600
plt.rcParams["figure.dpi"] = 120

# -----------------------------------------------------------------------------
# 2. Frozen definitions
# -----------------------------------------------------------------------------

STATE_COL = "final_malignant_state_id"
PATIENT_COL = "patient"

HARMONY_KEY = "X_pca_harmony_lineage_clean"
NEIGHBOR_KEY = "neighbors_lineage_clean"

EXPECTED_N_OBS = 12057
EXPECTED_N_VARS = 25712

EXPECTED_STATE_COUNTS = {
    "MS1": 1658,
    "MS2": 1277,
    "MS3": 720,
    "MS4": 2916,
    "MS5": 2392,
    "MS6": 557,
    "MS7": 881,
    "MS8": 1656,
}

FROZEN_STATE_NAMES = {
    "MS1": "Hypoxia-TGFb-Inflammatory",
    "MS2": "S100P-SPINK1-Epithelial",
    "MS3": "Cycling-G2M",
    "MS4": "HCC08-Replication-Fetal-Progenitor",
    "MS5": "Stress-Inflammatory-Progenitor",
    "MS6": "Hepatic-Differentiated",
    "MS7": "HCC04-Glycolytic-Hepatic",
    "MS8": "HCC04-Hepatic-BileAcid-Metabolic",
}

FROZEN_RULES = [
    "No reclustering.",
    "No malignant-state redefinition.",
    "No modification or reselection of MS3 Conserved Core50.",
    "No reconstruction of Original RiskScore in scRNA.",
    "MS1-MS8 labels are fixed.",
    "MS3 is the primary bulk-risk-concordant malignant state.",
    "MS6 is the primary bulk-risk-opposing malignant state.",
    "Primary biological contrast = MS3 vs MS6.",
    (
        "Trajectory analysis evaluates transcriptional continuity and "
        "pseudotemporal compatibility, not proven temporal evolution."
    ),
    "Patient composition must be explicitly considered.",
]

# -----------------------------------------------------------------------------
# 3. Environment audit
# -----------------------------------------------------------------------------

environment = {
    "Timestamp": datetime.now().isoformat(timespec="seconds"),
    "Python": sys.version.split()[0],
    "Platform": platform.platform(),
    "NumPy": np.__version__,
    "Pandas": pd.__version__,
    "SciPy": scipy.__version__,
    "Scanpy": sc.__version__,
    "AnnData": ad.__version__,
    "Random_seed": SEED,
}

print("=" * 100)
print("STEP27 CELL 1 — INITIALIZATION")
print("=" * 100)

print("\n[1] Environment")
for k, v in environment.items():
    print(f"{k:18s}: {v}")

# -----------------------------------------------------------------------------
# 4. Check frozen input
# -----------------------------------------------------------------------------

print("\n" + "=" * 100)
print("[2] Frozen input")
print("=" * 100)

assert INPUT_H5AD.exists(), f"Frozen input does not exist:\n{INPUT_H5AD}"

print("Input:")
print(INPUT_H5AD)

# Read only into memory.
# This does NOT modify the Step26 file unless adata.write_h5ad() is later called
# on that same path, which is prohibited in Step27.
adata = sc.read_h5ad(INPUT_H5AD)

print(f"\nAnnData shape: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

assert adata.n_obs == EXPECTED_N_OBS, (
    f"Cell-count mismatch: expected {EXPECTED_N_OBS}, got {adata.n_obs}"
)

assert adata.n_vars == EXPECTED_N_VARS, (
    f"Gene-count mismatch: expected {EXPECTED_N_VARS}, got {adata.n_vars}"
)

print("[PASS] Frozen AnnData dimensions verified.")

# -----------------------------------------------------------------------------
# 5. Essential metadata audit
# -----------------------------------------------------------------------------

print("\n" + "=" * 100)
print("[3] Essential metadata")
print("=" * 100)

assert STATE_COL in adata.obs.columns, (
    f"Missing frozen state column: {STATE_COL}"
)

assert PATIENT_COL in adata.obs.columns, (
    f"Missing patient column: {PATIENT_COL}"
)

assert HARMONY_KEY in adata.obsm.keys(), (
    f"Missing frozen Harmony representation: {HARMONY_KEY}"
)

print(f"State column   : {STATE_COL}")
print(f"Patient column : {PATIENT_COL}")
print(f"Harmony key    : {HARMONY_KEY}")
print(f"Harmony shape  : {adata.obsm[HARMONY_KEY].shape}")

assert adata.obsm[HARMONY_KEY].shape == (EXPECTED_N_OBS, 30), (
    "Unexpected Harmony representation shape."
)

print("[PASS] Essential frozen metadata verified.")

# -----------------------------------------------------------------------------
# 6. Frozen MS1-MS8 cell counts
# -----------------------------------------------------------------------------

print("\n" + "=" * 100)
print("[4] Frozen malignant-state counts")
print("=" * 100)

state_counts = (
    adata.obs[STATE_COL]
    .astype(str)
    .value_counts()
    .reindex(EXPECTED_STATE_COUNTS.keys())
    .fillna(0)
    .astype(int)
)

state_count_df = pd.DataFrame({
    "State": list(EXPECTED_STATE_COUNTS.keys()),
    "State_Name": [
        FROZEN_STATE_NAMES[x]
        for x in EXPECTED_STATE_COUNTS.keys()
    ],
    "Observed_N": [
        int(state_counts.loc[x])
        for x in EXPECTED_STATE_COUNTS.keys()
    ],
    "Expected_N": [
        EXPECTED_STATE_COUNTS[x]
        for x in EXPECTED_STATE_COUNTS.keys()
    ],
})

state_count_df["Match"] = (
    state_count_df["Observed_N"] ==
    state_count_df["Expected_N"]
)

display(state_count_df)

for state, expected_n in EXPECTED_STATE_COUNTS.items():
    observed_n = int(state_counts.loc[state])

    assert observed_n == expected_n, (
        f"{state} count mismatch: "
        f"expected {expected_n}, observed {observed_n}"
    )

assert state_count_df["Observed_N"].sum() == EXPECTED_N_OBS

print("\n[PASS] MS1-MS8 frozen counts are identical to the locked state universe.")

# -----------------------------------------------------------------------------
# 7. Patient × state composition
# -----------------------------------------------------------------------------

print("\n" + "=" * 100)
print("[5] Patient × malignant-state counts")
print("=" * 100)

patient_state_counts = pd.crosstab(
    adata.obs[PATIENT_COL].astype(str),
    adata.obs[STATE_COL].astype(str)
)

patient_state_counts = patient_state_counts.reindex(
    columns=list(EXPECTED_STATE_COUNTS.keys()),
    fill_value=0,
)

display(patient_state_counts)

# Critical within-patient MS3/MS6 validation patients
required_patient_counts = {
    "HCC09": {
        "MS3": 116,
        "MS6": 487,
    },
    "HCC10": {
        "MS3": 244,
        "MS6": 70,
    },
}

for patient, expected in required_patient_counts.items():

    assert patient in patient_state_counts.index, (
        f"{patient} not present in patient-state table."
    )

    for state, expected_n in expected.items():

        observed_n = int(
            patient_state_counts.loc[patient, state]
        )

        assert observed_n == expected_n, (
            f"{patient} {state} mismatch: "
            f"expected {expected_n}, got {observed_n}"
        )

print("\n[PASS] HCC09 contains MS3 + MS6: 116 / 487")
print("[PASS] HCC10 contains MS3 + MS6: 244 / 70")

# -----------------------------------------------------------------------------
# 8. Existing frozen neighbor graph
# -----------------------------------------------------------------------------

print("\n" + "=" * 100)
print("[6] Existing neighbor graph")
print("=" * 100)

print("uns keys containing 'neighbor':")
print([
    k for k in adata.uns.keys()
    if "neighbor" in str(k).lower()
])

print("\nobsp keys:")
print(list(adata.obsp.keys()))

assert NEIGHBOR_KEY in adata.uns, (
    f"Missing frozen neighbor metadata: {NEIGHBOR_KEY}"
)

neighbor_uns = adata.uns[NEIGHBOR_KEY]

print("\nNeighbor metadata:")
print(neighbor_uns)

neighbor_params = neighbor_uns.get("params", {})

print("\nNeighbor parameters:")
for k, v in neighbor_params.items():
    print(f"{k}: {v}")

# Expected frozen graph parameters
if "n_neighbors" in neighbor_params:
    assert int(neighbor_params["n_neighbors"]) == 15

if "n_pcs" in neighbor_params:
    assert int(neighbor_params["n_pcs"]) == 30

if "use_rep" in neighbor_params:
    assert neighbor_params["use_rep"] == HARMONY_KEY

if "metric" in neighbor_params:
    assert str(neighbor_params["metric"]).lower() == "euclidean"

connectivities_key = neighbor_uns.get(
    "connectivities_key",
    f"{NEIGHBOR_KEY}_connectivities"
)

distances_key = neighbor_uns.get(
    "distances_key",
    f"{NEIGHBOR_KEY}_distances"
)

print(f"\nConnectivities key: {connectivities_key}")
print(f"Distances key     : {distances_key}")

assert connectivities_key in adata.obsp.keys(), (
    f"Missing connectivities matrix: {connectivities_key}"
)

print(
    "Connectivities shape:",
    adata.obsp[connectivities_key].shape
)

print("[PASS] Existing frozen patient-only Harmony graph verified.")

# -----------------------------------------------------------------------------
# 9. Expression representation
# -----------------------------------------------------------------------------

print("\n" + "=" * 100)
print("[7] Expression representation")
print("=" * 100)

print("X type       :", type(adata.X))
print("raw exists   :", adata.raw is not None)
print("layers       :", list(adata.layers.keys()))
print("n genes      :", adata.n_vars)

# No raw counts are required for Step27 topology/pseudotime.
# The frozen object contains log-normalized expression.

# -----------------------------------------------------------------------------
# 10. MS3 Core50 audit
# -----------------------------------------------------------------------------

print("\n" + "=" * 100)
print("[8] MS3 Core50-related fields")
print("=" * 100)

core50_obs_cols = [
    c for c in adata.obs.columns
    if "core50" in str(c).lower()
]

core50_uns_keys = [
    k for k in adata.uns.keys()
    if "core50" in str(k).lower()
]

print("obs columns containing Core50:")
print(core50_obs_cols)

print("\nuns keys containing Core50:")
print(core50_uns_keys)

assert len(core50_obs_cols) > 0 or len(core50_uns_keys) > 0, (
    "No Core50-related field detected."
)

# If the frozen Core50 gene list exists in uns, inspect gene coverage.
core50_gene_list = None

for key in core50_uns_keys:

    obj = adata.uns[key]

    if isinstance(obj, (list, tuple, np.ndarray, pd.Index)):
        genes = [str(x) for x in obj]

        if len(genes) == 50:
            core50_gene_list = genes
            print(f"\nCore50 gene list detected in uns['{key}']")
            break

if core50_gene_list is not None:

    var_names = set(map(str, adata.var_names))

    core50_covered = [
        g for g in core50_gene_list
        if g in var_names
    ]

    print(
        f"Core50 coverage: "
        f"{len(core50_covered)}/{len(core50_gene_list)}"
    )

    assert len(core50_covered) == 50

else:
    print(
        "\nCore50 score/annotation exists, but the exact 50-gene list "
        "was not recovered automatically from .uns in this cell."
    )
    print(
        "Preflight audit already verified frozen Core50 coverage = 50/50."
    )

# -----------------------------------------------------------------------------
# 11. Step26 biological-program annotations
# -----------------------------------------------------------------------------

print("\n" + "=" * 100)
print("[9] Step26 biological-program annotations")
print("=" * 100)

program_keywords = [
    "mito",
    "mitotic",
    "g2m",
    "cin",
    "chromosome",
    "replication",
    "repair",
    "fanconi",
    "homolog",
    "nucleotide",
    "rna",
    "splice",
    "ribosome",
    "hepatic",
    "bile",
    "xenobiotic",
    "core50",
]

program_cols = []

for c in adata.obs.columns:
    cl = str(c).lower()

    if any(k in cl for k in program_keywords):
        program_cols.append(c)

print(f"Detected candidate Step26 program columns: {len(program_cols)}")

for c in program_cols:
    print(" -", c)

# This is intentionally an audit rather than a hard-coded requirement,
# because exact Step26 score-column names are frozen from upstream analysis.

# -----------------------------------------------------------------------------
# 12. Step28 RNA velocity feasibility audit
# -----------------------------------------------------------------------------

print("\n" + "=" * 100)
print("[10] RNA velocity layer audit")
print("=" * 100)

velocity_layers = [
    x for x in [
        "spliced",
        "unspliced",
        "ambiguous",
    ]
    if x in adata.layers.keys()
]

print("Velocity-related layers:", velocity_layers)

if not (
    "spliced" in adata.layers.keys()
    and "unspliced" in adata.layers.keys()
):
    print("RNA_VELOCITY_INPUT_NOT_PRESENT")

# -----------------------------------------------------------------------------
# 13. Save Cell-1 audit tables
# -----------------------------------------------------------------------------

state_count_path = (
    TABLE_DIR /
    "Step27_Cell1_Frozen_State_Counts.csv"
)

patient_state_path = (
    TABLE_DIR /
    "Step27_Cell1_Patient_State_Counts.csv"
)

if not state_count_path.exists():
    state_count_df.to_csv(
        state_count_path,
        index=False,
        encoding="utf-8-sig",
    )
    print(f"\nSaved: {state_count_path}")
else:
    print(f"\nExists, not overwritten: {state_count_path}")

if not patient_state_path.exists():
    patient_state_counts.to_csv(
        patient_state_path,
        encoding="utf-8-sig",
    )
    print(f"Saved: {patient_state_path}")
else:
    print(f"Exists, not overwritten: {patient_state_path}")

# -----------------------------------------------------------------------------
# 14. Save initialization manifest
# -----------------------------------------------------------------------------

initialization = {
    "Step": "Step27",
    "Stage": "Cell1_Initialization_FrozenInputVerification",
    "Timestamp": datetime.now().isoformat(timespec="seconds"),
    "Input_H5AD": str(INPUT_H5AD),
    "N_obs": int(adata.n_obs),
    "N_vars": int(adata.n_vars),
    "State_column": STATE_COL,
    "Patient_column": PATIENT_COL,
    "Harmony_key": HARMONY_KEY,
    "Harmony_dimensions": int(
        adata.obsm[HARMONY_KEY].shape[1]
    ),
    "Neighbor_key": NEIGHBOR_KEY,
    "Connectivities_key": connectivities_key,
    "Distances_key": distances_key,
    "Neighbor_params": {
        str(k): (
            v.item()
            if isinstance(v, np.generic)
            else v
        )
        for k, v in neighbor_params.items()
    },
    "Expected_state_counts": EXPECTED_STATE_COUNTS,
    "Frozen_state_names": FROZEN_STATE_NAMES,
    "Core50_audit": {
        "Detected_obs_columns": [
            str(x) for x in core50_obs_cols
        ],
        "Detected_uns_keys": [
            str(x) for x in core50_uns_keys
        ],
        "Preflight_coverage": "50/50",
    },
    "RNA_velocity_input_present": bool(
        "spliced" in adata.layers.keys()
        and "unspliced" in adata.layers.keys()
    ),
    "Frozen_rules": FROZEN_RULES,
    "Environment": environment,
}

initialization_path = (
    MANIFEST_DIR /
    "Step27_Initialization.json"
)

if not initialization_path.exists():

    with open(
        initialization_path,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            initialization,
            f,
            indent=2,
            ensure_ascii=False,
            default=str,
        )

    print(f"Saved: {initialization_path}")

else:
    print(
        f"Exists, not overwritten: "
        f"{initialization_path}"
    )

# -----------------------------------------------------------------------------
# 15. Final Cell-1 status
# -----------------------------------------------------------------------------

print("\n" + "=" * 100)
print("STEP27 CELL 1 COMPLETE")
print("=" * 100)

print(f"Frozen input cells : {adata.n_obs:,}")
print(f"Frozen genes       : {adata.n_vars:,}")
print(f"Frozen states      : {adata.obs[STATE_COL].nunique()}")
print(f"Patients           : {adata.obs[PATIENT_COL].nunique()}")
print(f"Harmony dimensions : {adata.obsm[HARMONY_KEY].shape[1]}")
print(f"Core50 coverage    : 50/50")
print(
    "RNA velocity      :",
    "AVAILABLE"
    if initialization["RNA_velocity_input_present"]
    else "NOT PRESENT"
)

print(
    "\nSTEP27 CELL 1 COMPLETE — "
    "FROZEN INPUT VERIFIED — "
    "NO TRAJECTORY ANALYSIS EXECUTED"
)

In [ ]:
# =============================================================================
# Step27 Cell 2
# Step27B — Global Malignant-State Topology Using Frozen Harmony Graph
# =============================================================================

from pathlib import Path
import inspect
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse

# -----------------------------------------------------------------------------
# 0. Reconfirm required objects from Cell 1
# -----------------------------------------------------------------------------

required_globals = [
    "adata",
    "STATE_COL",
    "PATIENT_COL",
    "NEIGHBOR_KEY",
    "TABLE_DIR",
    "FIGURE_DIR",
    "MANIFEST_DIR",
]

missing_globals = [
    x for x in required_globals
    if x not in globals()
]

assert len(missing_globals) == 0, (
    "Missing required objects from Cell 1: "
    + ", ".join(missing_globals)
)

STATE_ORDER = [
    "MS1",
    "MS2",
    "MS3",
    "MS4",
    "MS5",
    "MS6",
    "MS7",
    "MS8",
]

print("=" * 110)
print("STEP27 CELL 2 — GLOBAL MALIGNANT-STATE TOPOLOGY")
print("=" * 110)

print("\nAnalysis principles:")
print("- Frozen MS1-MS8 labels only")
print("- Existing patient-only Harmony graph only")
print("- No reclustering")
print("- No new neighbors")
print("- No root")
print("- No pseudotime")
print("- No directionality assumption")

# -----------------------------------------------------------------------------
# 1. Create Step27 working copy
# -----------------------------------------------------------------------------

# IMPORTANT:
# All Step27 computations are performed on an in-memory copy.
# The frozen Step26 source file is never written back.

adata27 = adata.copy()

adata27.obs[STATE_COL] = pd.Categorical(
    adata27.obs[STATE_COL].astype(str),
    categories=STATE_ORDER,
    ordered=True,
)

assert adata27.obs[STATE_COL].isna().sum() == 0
assert list(adata27.obs[STATE_COL].cat.categories) == STATE_ORDER

print("\n[1] Working copy created")
print(f"Cells : {adata27.n_obs:,}")
print(f"Genes : {adata27.n_vars:,}")
print(f"Groups: {list(adata27.obs[STATE_COL].cat.categories)}")

# -----------------------------------------------------------------------------
# 2. Reconfirm frozen graph
# -----------------------------------------------------------------------------

print("\n" + "=" * 110)
print("[2] Frozen graph verification")
print("=" * 110)

assert NEIGHBOR_KEY in adata27.uns

neighbor_info = adata27.uns[NEIGHBOR_KEY]

connectivities_key = neighbor_info["connectivities_key"]
distances_key = neighbor_info["distances_key"]

assert connectivities_key in adata27.obsp
assert distances_key in adata27.obsp

frozen_connectivities = adata27.obsp[connectivities_key]

print(f"Neighbor key       : {NEIGHBOR_KEY}")
print(f"Connectivities key : {connectivities_key}")
print(f"Distances key      : {distances_key}")
print(f"Graph shape        : {frozen_connectivities.shape}")
print(f"Graph nonzero      : {frozen_connectivities.nnz:,}")

print("\nFrozen neighbor parameters:")
for k, v in neighbor_info.get("params", {}).items():
    print(f"  {k}: {v}")

# -----------------------------------------------------------------------------
# 3. Run PAGA using EXISTING frozen neighbor graph
# -----------------------------------------------------------------------------

print("\n" + "=" * 110)
print("[3] PAGA on frozen MS1-MS8 states")
print("=" * 110)

# This writes only to adata27.uns['paga'] in memory.
# It does NOT modify the original frozen h5ad.
sc.tl.paga(
    adata27,
    groups=STATE_COL,
    model="v1.2",
    neighbors_key=NEIGHBOR_KEY,
)

assert "paga" in adata27.uns

print("PAGA groups:")
print(adata27.uns["paga"]["groups"])

print("\nPAGA result keys:")
print(list(adata27.uns["paga"].keys()))

# -----------------------------------------------------------------------------
# 4. Extract PAGA connectivity matrix
# -----------------------------------------------------------------------------

print("\n" + "=" * 110)
print("[4] PAGA connectivity matrix")
print("=" * 110)

paga_conn = adata27.uns["paga"]["connectivities"]

if sparse.issparse(paga_conn):
    paga_conn_array = paga_conn.toarray()
else:
    paga_conn_array = np.asarray(paga_conn)

assert paga_conn_array.shape == (8, 8)

paga_conn_df = pd.DataFrame(
    paga_conn_array,
    index=STATE_ORDER,
    columns=STATE_ORDER,
)

display(
    paga_conn_df.round(4)
)

# -----------------------------------------------------------------------------
# 5. Long-format edge table
# -----------------------------------------------------------------------------

edge_records = []

for i in range(len(STATE_ORDER)):
    for j in range(i + 1, len(STATE_ORDER)):

        s1 = STATE_ORDER[i]
        s2 = STATE_ORDER[j]

        weight = float(
            paga_conn_df.loc[s1, s2]
        )

        edge_records.append(
            {
                "State_1": s1,
                "State_2": s2,
                "PAGA_connectivity": weight,
                "Positive_edge": weight > 0,
            }
        )

edge_df = pd.DataFrame(edge_records)

edge_df = edge_df.sort_values(
    "PAGA_connectivity",
    ascending=False,
).reset_index(drop=True)

print("\n" + "=" * 110)
print("[5] All pairwise PAGA edges — ranked")
print("=" * 110)

display(
    edge_df.round(4)
)

# -----------------------------------------------------------------------------
# 6. Extract PAGA tree-like subgraph
# -----------------------------------------------------------------------------

print("\n" + "=" * 110)
print("[6] PAGA tree-like topology")
print("=" * 110)

tree_edges_df = pd.DataFrame()

if "connectivities_tree" in adata27.uns["paga"]:

    tree_conn = adata27.uns["paga"]["connectivities_tree"]

    if sparse.issparse(tree_conn):
        tree_array = tree_conn.toarray()
    else:
        tree_array = np.asarray(tree_conn)

    tree_records = []

    for i in range(len(STATE_ORDER)):
        for j in range(i + 1, len(STATE_ORDER)):

            weight = float(tree_array[i, j])

            if weight > 0:
                tree_records.append(
                    {
                        "State_1": STATE_ORDER[i],
                        "State_2": STATE_ORDER[j],
                        "Tree_connectivity": weight,
                    }
                )

    tree_edges_df = pd.DataFrame(tree_records)

    if len(tree_edges_df) > 0:
        tree_edges_df = tree_edges_df.sort_values(
            "Tree_connectivity",
            ascending=False,
        ).reset_index(drop=True)

    display(tree_edges_df.round(4))

else:
    print("No connectivities_tree present.")

# -----------------------------------------------------------------------------
# 7. State-level graph summary
# -----------------------------------------------------------------------------

print("\n" + "=" * 110)
print("[7] State-level topology summary")
print("=" * 110)

state_graph_records = []

for state in STATE_ORDER:

    weights = paga_conn_df.loc[state].copy()

    # Remove self
    weights = weights.drop(state)

    positive_weights = weights[
        weights > 0
    ].sort_values(ascending=False)

    strongest_state = (
        positive_weights.index[0]
        if len(positive_weights) > 0
        else None
    )

    strongest_weight = (
        float(positive_weights.iloc[0])
        if len(positive_weights) > 0
        else np.nan
    )

    total_connectivity = float(
        positive_weights.sum()
    )

    n_positive_edges = int(
        (weights > 0).sum()
    )

    state_graph_records.append(
        {
            "State": state,
            "State_Name": FROZEN_STATE_NAMES[state],
            "N_cells": int(
                (adata27.obs[STATE_COL] == state).sum()
            ),
            "N_positive_PAGA_edges": n_positive_edges,
            "Total_PAGA_connectivity": total_connectivity,
            "Strongest_connected_state": strongest_state,
            "Strongest_edge_weight": strongest_weight,
        }
    )

state_graph_summary = pd.DataFrame(
    state_graph_records
)

display(
    state_graph_summary.round(4)
)

# -----------------------------------------------------------------------------
# 8. Focused MS3 / MS6 connectivity
# -----------------------------------------------------------------------------

print("\n" + "=" * 110)
print("[8] MS3 / MS6 focused topology")
print("=" * 110)

ms3_ms6_direct = float(
    paga_conn_df.loc["MS3", "MS6"]
)

print(
    f"Direct PAGA connectivity MS3 ↔ MS6: "
    f"{ms3_ms6_direct:.6f}"
)

def show_state_neighbors(state, top_n=7):

    x = (
        paga_conn_df.loc[state]
        .drop(state)
        .sort_values(ascending=False)
    )

    out = pd.DataFrame(
        {
            "State": x.index,
            "PAGA_connectivity": x.values,
        }
    )

    out["From_state"] = state

    return out[
        [
            "From_state",
            "State",
            "PAGA_connectivity",
        ]
    ].head(top_n)

ms3_neighbors = show_state_neighbors(
    "MS3"
)

ms6_neighbors = show_state_neighbors(
    "MS6"
)

print("\nMS3 connections:")
display(
    ms3_neighbors.round(4)
)

print("\nMS6 connections:")
display(
    ms6_neighbors.round(4)
)

# -----------------------------------------------------------------------------
# 9. Explicit MS1 / MS5 positioning
# -----------------------------------------------------------------------------

print("\n" + "=" * 110)
print("[9] MS1 / MS5 topology")
print("=" * 110)

ms1_neighbors = show_state_neighbors(
    "MS1"
)

ms5_neighbors = show_state_neighbors(
    "MS5"
)

print("\nMS1 connections:")
display(
    ms1_neighbors.round(4)
)

print("\nMS5 connections:")
display(
    ms5_neighbors.round(4)
)

# -----------------------------------------------------------------------------
# 10. Save topology tables
# -----------------------------------------------------------------------------

paga_matrix_path = (
    TABLE_DIR /
    "Step27B_PAGA_Connectivity_Matrix.csv"
)

edge_table_path = (
    TABLE_DIR /
    "Step27B_PAGA_AllEdges.csv"
)

tree_edge_path = (
    TABLE_DIR /
    "Step27B_PAGA_TreeEdges.csv"
)

state_summary_path = (
    TABLE_DIR /
    "Step27B_State_Topology_Summary.csv"
)

ms3_neighbor_path = (
    TABLE_DIR /
    "Step27B_MS3_Connectivity.csv"
)

ms6_neighbor_path = (
    TABLE_DIR /
    "Step27B_MS6_Connectivity.csv"
)

output_tables = {
    paga_matrix_path: paga_conn_df,
    edge_table_path: edge_df,
    state_summary_path: state_graph_summary,
    ms3_neighbor_path: ms3_neighbors,
    ms6_neighbor_path: ms6_neighbors,
}

if len(tree_edges_df) > 0:
    output_tables[
        tree_edge_path
    ] = tree_edges_df

for path, df in output_tables.items():

    if not path.exists():

        df.to_csv(
            path,
            encoding="utf-8-sig",
            index=(
                path == paga_matrix_path
            ),
        )

        print(f"Saved: {path}")

    else:
        print(
            f"Exists, not overwritten: {path}"
        )

# -----------------------------------------------------------------------------
# 11. Figure 1 — PAGA graph, all edges
# -----------------------------------------------------------------------------

print("\n" + "=" * 110)
print("[10] Saving global topology figures")
print("=" * 110)

fig, ax = plt.subplots(
    figsize=(8, 7)
)

# Compatibility across Scanpy versions:
paga_plot_signature = inspect.signature(
    sc.pl.paga
)

paga_plot_kwargs = dict(
    adata=adata27,
    threshold=0.0,
    color=STATE_COL,
    labels=STATE_ORDER,
    layout="fr",
    node_size_scale=1.8,
    edge_width_scale=3.0,
    fontsize=11,
    frameon=False,
    show=False,
    ax=ax,
)

if "rng" in paga_plot_signature.parameters:
    paga_plot_kwargs["rng"] = 42

elif "random_state" in paga_plot_signature.parameters:
    paga_plot_kwargs["random_state"] = 42

sc.pl.paga(
    **paga_plot_kwargs
)

ax.set_title(
    "Global Malignant-State Topology",
    fontsize=14,
)

fig.tight_layout()

paga_png = (
    FIGURE_DIR /
    "Step27B_01_Global_PAGA_AllEdges.png"
)

paga_pdf = (
    FIGURE_DIR /
    "Step27B_01_Global_PAGA_AllEdges.pdf"
)

fig.savefig(
    paga_png,
    dpi=600,
    bbox_inches="tight",
)

fig.savefig(
    paga_pdf,
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

print(f"Saved: {paga_png}")
print(f"Saved: {paga_pdf}")

# -----------------------------------------------------------------------------
# 12. Figure 2 — PAGA connectivity matrix
# -----------------------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(7.2, 6.2)
)

im = ax.imshow(
    paga_conn_df.values,
    aspect="equal",
)

ax.set_xticks(
    np.arange(len(STATE_ORDER))
)

ax.set_xticklabels(
    STATE_ORDER,
    rotation=45,
    ha="right",
)

ax.set_yticks(
    np.arange(len(STATE_ORDER))
)

ax.set_yticklabels(
    STATE_ORDER
)

ax.set_title(
    "PAGA Connectivity Matrix",
    fontsize=14,
)

cbar = fig.colorbar(
    im,
    ax=ax,
    fraction=0.046,
    pad=0.04,
)

cbar.set_label(
    "Connectivity"
)

# Annotate values
for i in range(len(STATE_ORDER)):
    for j in range(len(STATE_ORDER)):

        value = paga_conn_df.iloc[i, j]

        ax.text(
            j,
            i,
            f"{value:.2f}",
            ha="center",
            va="center",
            fontsize=7,
        )

fig.tight_layout()

heatmap_png = (
    FIGURE_DIR /
    "Step27B_02_PAGA_Connectivity_Matrix.png"
)

heatmap_pdf = (
    FIGURE_DIR /
    "Step27B_02_PAGA_Connectivity_Matrix.pdf"
)

fig.savefig(
    heatmap_png,
    dpi=600,
    bbox_inches="tight",
)

fig.savefig(
    heatmap_pdf,
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

print(f"Saved: {heatmap_png}")
print(f"Saved: {heatmap_pdf}")

# -----------------------------------------------------------------------------
# 13. Figure 3 — PAGA edge-weight distribution
# -----------------------------------------------------------------------------

positive_edges = edge_df.loc[
    edge_df["PAGA_connectivity"] > 0,
    "PAGA_connectivity",
].values

fig, ax = plt.subplots(
    figsize=(7, 5)
)

ax.hist(
    positive_edges,
    bins=min(
        15,
        max(
            5,
            len(positive_edges)
        )
    ),
)

ax.set_xlabel(
    "PAGA connectivity"
)

ax.set_ylabel(
    "Number of state pairs"
)

ax.set_title(
    "Distribution of Positive PAGA Edge Weights"
)

fig.tight_layout()

hist_png = (
    FIGURE_DIR /
    "Step27B_03_PAGA_EdgeWeight_Distribution.png"
)

hist_pdf = (
    FIGURE_DIR /
    "Step27B_03_PAGA_EdgeWeight_Distribution.pdf"
)

fig.savefig(
    hist_png,
    dpi=600,
    bbox_inches="tight",
)

fig.savefig(
    hist_pdf,
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

print(f"Saved: {hist_png}")
print(f"Saved: {hist_pdf}")

# -----------------------------------------------------------------------------
# 14. Save Step27B summary
# -----------------------------------------------------------------------------

def dataframe_records(df):
    return json.loads(
        df.to_json(
            orient="records"
        )
    )

step27b_summary = {
    "Step": "Step27B",
    "Analysis": "Global malignant-state topology",
    "Input_cells": int(adata27.n_obs),
    "Frozen_state_column": STATE_COL,
    "Neighbor_key": NEIGHBOR_KEY,
    "PAGA_model": "v1.2",
    "Root_defined": False,
    "Pseudotime_computed": False,
    "New_neighbors_computed": False,
    "Reclustering_performed": False,
    "MS3_MS6_direct_PAGA_connectivity": (
        ms3_ms6_direct
    ),
    "MS3_connectivity": dataframe_records(
        ms3_neighbors
    ),
    "MS6_connectivity": dataframe_records(
        ms6_neighbors
    ),
    "MS1_connectivity": dataframe_records(
        ms1_neighbors
    ),
    "MS5_connectivity": dataframe_records(
        ms5_neighbors
    ),
    "Interpretation_boundary": (
        "PAGA connectivity describes coarse-grained "
        "topological connectivity among frozen malignant states. "
        "It does not establish temporal direction, lineage "
        "conversion, or actual evolutionary transitions."
    ),
}

summary_path = (
    MANIFEST_DIR /
    "Step27B_GlobalTopology_Summary.json"
)

if not summary_path.exists():

    with open(
        summary_path,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            step27b_summary,
            f,
            indent=2,
            ensure_ascii=False,
        )

    print(f"Saved: {summary_path}")

else:
    print(
        f"Exists, not overwritten: "
        f"{summary_path}"
    )

# -----------------------------------------------------------------------------
# 15. Cell status
# -----------------------------------------------------------------------------

print("\n" + "=" * 110)
print("STEP27 CELL 2 COMPLETE")
print("=" * 110)

print(
    f"MS3 ↔ MS6 direct PAGA connectivity: "
    f"{ms3_ms6_direct:.6f}"
)

print(
    "Strongest MS3 connection:",
    ms3_neighbors.iloc[0]["State"],
    f"({ms3_neighbors.iloc[0]['PAGA_connectivity']:.6f})",
)

print(
    "Strongest MS6 connection:",
    ms6_neighbors.iloc[0]["State"],
    f"({ms6_neighbors.iloc[0]['PAGA_connectivity']:.6f})",
)

print(
    "\nSTEP27 CELL 2 COMPLETE — "
    "GLOBAL TOPOLOGY COMPUTED — "
    "NO ROOT / NO PSEUDOTIME / NO DIRECTION ASSUMED"
)

In [ ]:
# =============================================================================
# Step27 Cell 2
# Global topology — prepare working copy and compute PAGA
# =============================================================================

import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

STATE_ORDER = [
    "MS1", "MS2", "MS3", "MS4",
    "MS5", "MS6", "MS7", "MS8"
]

print("=" * 90)
print("STEP27 CELL 2 — COMPUTE PAGA ON FROZEN HARMONY GRAPH")
print("=" * 90)

# -----------------------------------------------------------------------------
# 1. Create an in-memory Step27 working copy
# -----------------------------------------------------------------------------

adata27 = adata.copy()

adata27.obs[STATE_COL] = pd.Categorical(
    adata27.obs[STATE_COL].astype(str),
    categories=STATE_ORDER,
    ordered=True
)

assert adata27.obs[STATE_COL].isna().sum() == 0

print(f"Cells : {adata27.n_obs:,}")
print(f"Genes : {adata27.n_vars:,}")
print(f"States: {list(adata27.obs[STATE_COL].cat.categories)}")

# -----------------------------------------------------------------------------
# 2. Verify frozen neighbor graph
# -----------------------------------------------------------------------------

assert NEIGHBOR_KEY in adata27.uns

neighbor_info = adata27.uns[NEIGHBOR_KEY]

connectivities_key = neighbor_info["connectivities_key"]
distances_key = neighbor_info["distances_key"]

assert connectivities_key in adata27.obsp
assert distances_key in adata27.obsp

print("\nFrozen graph:")
print("Neighbor key      :", NEIGHBOR_KEY)
print("Connectivities key:", connectivities_key)
print("Distances key     :", distances_key)

print("\nNeighbor parameters:")
for k, v in neighbor_info.get("params", {}).items():
    print(f"  {k}: {v}")

# -----------------------------------------------------------------------------
# 3. Compute PAGA using the EXISTING frozen graph
# -----------------------------------------------------------------------------

sc.tl.paga(
    adata27,
    groups=STATE_COL,
    model="v1.2",
    neighbors_key=NEIGHBOR_KEY
)

assert "paga" in adata27.uns

print("\nPAGA result keys:")
print(list(adata27.uns["paga"].keys()))

print("\nPAGA groups:")
print(adata27.uns["paga"]["groups"])

print("\nSTEP27 CELL 2 COMPLETE")
print("PAGA computed using frozen patient-only Harmony graph.")
print("NO reclustering / NO new neighbors / NO root / NO pseudotime.")

In [ ]:
# =============================================================================
# Step27 Cell 3
# Extract and inspect PAGA connectivity matrix
# =============================================================================

from scipy import sparse
import numpy as np
import pandas as pd

paga_conn = adata27.uns["paga"]["connectivities"]

if sparse.issparse(paga_conn):
    paga_conn = paga_conn.toarray()

paga_conn_df = pd.DataFrame(
    paga_conn,
    index=STATE_ORDER,
    columns=STATE_ORDER
)

display(paga_conn_df.round(4))

In [ ]:
# =============================================================================
# Step27 Cell 4
# Rank all state-state PAGA edges
# =============================================================================

edge_records = []

for i in range(len(STATE_ORDER)):
    for j in range(i + 1, len(STATE_ORDER)):
        s1 = STATE_ORDER[i]
        s2 = STATE_ORDER[j]

        edge_records.append({
            "State_1": s1,
            "State_2": s2,
            "PAGA_connectivity": float(paga_conn_df.loc[s1, s2])
        })

edge_df = pd.DataFrame(edge_records)

edge_df = edge_df.sort_values(
    "PAGA_connectivity",
    ascending=False
).reset_index(drop=True)

display(edge_df.round(4))

In [ ]:
# =============================================================================
# Step27 Cell 5
# Extract PAGA tree topology
# =============================================================================

tree_conn = adata27.uns["paga"]["connectivities_tree"]

if sparse.issparse(tree_conn):
    tree_conn = tree_conn.toarray()

tree_records = []

for i in range(len(STATE_ORDER)):
    for j in range(i + 1, len(STATE_ORDER)):
        weight = float(tree_conn[i, j])

        if weight > 0:
            tree_records.append({
                "State_1": STATE_ORDER[i],
                "State_2": STATE_ORDER[j],
                "Tree_connectivity": weight
            })

tree_edges_df = pd.DataFrame(tree_records)

tree_edges_df = tree_edges_df.sort_values(
    "Tree_connectivity",
    ascending=False
).reset_index(drop=True)

display(tree_edges_df.round(4))

In [ ]:
# =============================================================================
# Step27 Cell 6
# Focused topology: MS3 and MS6
# =============================================================================

def get_state_neighbors(state):
    x = (
        paga_conn_df.loc[state]
        .drop(state)
        .sort_values(ascending=False)
    )

    return pd.DataFrame({
        "State": x.index,
        "PAGA_connectivity": x.values
    })

ms3_neighbors = get_state_neighbors("MS3")
ms6_neighbors = get_state_neighbors("MS6")

print("MS3 connections:")
display(ms3_neighbors.round(4))

print("\nMS6 connections:")
display(ms6_neighbors.round(4))

ms3_ms6_direct = float(
    paga_conn_df.loc["MS3", "MS6"]
)

print(
    "\nDirect MS3 ↔ MS6 connectivity:",
    round(ms3_ms6_direct, 4)
)

In [ ]:
# =============================================================================
# Step27 Cell 7
# Focused topology: MS1 and MS5
# =============================================================================

ms1_neighbors = get_state_neighbors("MS1")
ms5_neighbors = get_state_neighbors("MS5")

print("MS1 connections:")
display(ms1_neighbors.round(4))

print("\nMS5 connections:")
display(ms5_neighbors.round(4))

In [ ]:
# =============================================================================
# Step27 Cell 8
# Plot global PAGA topology
# =============================================================================

import matplotlib.pyplot as plt

sc.pl.paga(
    adata27,
    color=STATE_COL,
    threshold=0.05,
    layout="fr",
    node_size_scale=1.8,
    edge_width_scale=3,
    fontsize=11,
    frameon=False,
    show=False
)

fig = plt.gcf()

plt.title(
    "Global Malignant-State Topology",
    fontsize=14
)

plt.tight_layout()

png_path = FIGURE_DIR / "Step27B_Global_PAGA_Topology.png"
pdf_path = FIGURE_DIR / "Step27B_Global_PAGA_Topology.pdf"

fig.savefig(
    png_path,
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    pdf_path,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

print("Saved:")
print(png_path)
print(pdf_path)

In [ ]:
# =============================================================================
# Step27 Cell 9
# PAGA connectivity heatmap
# =============================================================================

fig, ax = plt.subplots(
    figsize=(7, 6)
)

im = ax.imshow(
    paga_conn_df.values,
    aspect="equal"
)

ax.set_xticks(
    range(len(STATE_ORDER))
)

ax.set_xticklabels(
    STATE_ORDER,
    rotation=45,
    ha="right"
)

ax.set_yticks(
    range(len(STATE_ORDER))
)

ax.set_yticklabels(
    STATE_ORDER
)

for i in range(len(STATE_ORDER)):
    for j in range(len(STATE_ORDER)):
        ax.text(
            j,
            i,
            f"{paga_conn_df.iloc[i, j]:.2f}",
            ha="center",
            va="center",
            fontsize=8
        )

ax.set_title(
    "PAGA Connectivity Matrix",
    fontsize=14
)

cbar = fig.colorbar(
    im,
    ax=ax
)

cbar.set_label(
    "PAGA connectivity"
)

plt.tight_layout()

png_path = FIGURE_DIR / "Step27B_PAGA_Connectivity_Heatmap.png"
pdf_path = FIGURE_DIR / "Step27B_PAGA_Connectivity_Heatmap.pdf"

fig.savefig(
    png_path,
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    pdf_path,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

print("Saved:")
print(png_path)
print(pdf_path)

In [ ]:
# =============================================================================
# Step27 Cell 10
# Save Step27B topology results
# =============================================================================

import json

# ------------------------------------------------------------------
# Save tables
# ------------------------------------------------------------------

paga_conn_df.to_csv(
    TABLE_DIR / "Step27B_PAGA_Connectivity_Matrix.csv",
    encoding="utf-8-sig"
)

edge_df.to_csv(
    TABLE_DIR / "Step27B_PAGA_AllEdges.csv",
    index=False,
    encoding="utf-8-sig"
)

tree_edges_df.to_csv(
    TABLE_DIR / "Step27B_PAGA_TreeEdges.csv",
    index=False,
    encoding="utf-8-sig"
)

ms3_neighbors.to_csv(
    TABLE_DIR / "Step27B_MS3_Connectivity.csv",
    index=False,
    encoding="utf-8-sig"
)

ms6_neighbors.to_csv(
    TABLE_DIR / "Step27B_MS6_Connectivity.csv",
    index=False,
    encoding="utf-8-sig"
)

ms1_neighbors.to_csv(
    TABLE_DIR / "Step27B_MS1_Connectivity.csv",
    index=False,
    encoding="utf-8-sig"
)

ms5_neighbors.to_csv(
    TABLE_DIR / "Step27B_MS5_Connectivity.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------------
# Summary
# ------------------------------------------------------------------

step27b_summary = {
    "Step": "Step27B",
    "Analysis": "Global malignant-state topology",

    "Input_cells": int(adata27.n_obs),

    "State_column": STATE_COL,

    "Neighbor_key": NEIGHBOR_KEY,

    "MS3_MS6_direct_connectivity": float(
        paga_conn_df.loc["MS3", "MS6"]
    ),

    "Strongest_MS3_connection": {
        "State": str(ms3_neighbors.iloc[0]["State"]),
        "Connectivity": float(
            ms3_neighbors.iloc[0]["PAGA_connectivity"]
        )
    },

    "Strongest_MS6_connection": {
        "State": str(ms6_neighbors.iloc[0]["State"]),
        "Connectivity": float(
            ms6_neighbors.iloc[0]["PAGA_connectivity"]
        )
    },

    "Strongest_MS5_connection": {
        "State": str(ms5_neighbors.iloc[0]["State"]),
        "Connectivity": float(
            ms5_neighbors.iloc[0]["PAGA_connectivity"]
        )
    },

    "Root_defined": False,
    "Pseudotime_computed": False,
    "Direction_assumed": False,

    "Interpretation_boundary": (
        "PAGA represents coarse-grained graph connectivity "
        "among frozen malignant states and does not establish "
        "temporal evolution or lineage conversion."
    )
}

summary_path = (
    MANIFEST_DIR /
    "Step27B_GlobalTopology_Summary.json"
)

with open(
    summary_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        step27b_summary,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Step27B tables and summary saved.")

print("\nKey result:")
print(
    "MS3 ↔ MS6 =",
    round(
        paga_conn_df.loc["MS3", "MS6"],
        4
    )
)

print(
    "Strongest MS3 connection =",
    ms3_neighbors.iloc[0]["State"]
)

print(
    "Strongest MS6 connection =",
    ms6_neighbors.iloc[0]["State"]
)

print(
    "\nSTEP27B COMPLETE — "
    "GLOBAL TOPOLOGY ONLY — "
    "NO PSEUDOTIME"
)

In [ ]:
# =============================================================================
# Step27 Cell 11
# Step27C-1 — Diffusion map on frozen Harmony graph
# =============================================================================

sc.tl.diffmap(
    adata27,
    n_comps=15,
    neighbors_key=NEIGHBOR_KEY,
    random_state=42
)

print("Diffusion map completed.")
print("X_diffmap shape:", adata27.obsm["X_diffmap"].shape)

In [ ]:
# =============================================================================
# Step27 Cell 12
# REPORT — Diffusion map audit
# =============================================================================

print("=" * 80)
print("STEP27C-1 REPORT — DIFFUSION MAP")
print("=" * 80)

print("X_diffmap exists:",
      "X_diffmap" in adata27.obsm)

print("Shape:",
      adata27.obsm["X_diffmap"].shape)

print("\nFirst 5 cells, first 6 diffusion components:")

display(
    pd.DataFrame(
        adata27.obsm["X_diffmap"][:, :6],
        columns=[
            f"DC{i+1}"
            for i in range(6)
        ],
        index=adata27.obs_names
    ).head()
)

print("\nNaN count:",
      np.isnan(
          adata27.obsm["X_diffmap"]
      ).sum())

print("\nSTEP27C-1 REPORT COMPLETE")

In [ ]:
# =============================================================================
# Step27 Cell 13
# State-level diffusion component summary
# =============================================================================

for i in range(1, 7):
    adata27.obs[f"DC{i}"] = (
        adata27.obsm["X_diffmap"][:, i - 1]
    )

dc_state_summary = (
    adata27.obs
    .groupby(
        STATE_COL,
        observed=True
    )[
        [f"DC{i}" for i in range(1, 7)]
    ]
    .agg(["median", "mean"])
)

display(dc_state_summary.round(4))

In [ ]:
# =============================================================================
# Step27 Cell 14
# REPORT — Diffusion coordinates of MS6/MS5/MS3
# =============================================================================

focus_states = [
    "MS6",
    "MS5",
    "MS3",
    "MS2",
    "MS1"
]

focus_dc_summary = (
    adata27.obs.loc[
        adata27.obs[STATE_COL].isin(
            focus_states
        )
    ]
    .groupby(
        STATE_COL,
        observed=True
    )[
        [f"DC{i}" for i in range(1, 7)]
    ]
    .agg(
        ["median", "mean", "std"]
    )
)

display(
    focus_dc_summary.round(4)
)

print("\nREPORT COMPLETE")

In [ ]:
# =============================================================================
# Step27 Cell 15
# Plot DC1 vs DC2 by malignant state
# =============================================================================

fig, ax = plt.subplots(
    figsize=(7, 6)
)

sc.pl.embedding(
    adata27,
    basis="diffmap",
    color=STATE_COL,
    components="1,2",
    size=12,
    alpha=0.7,
    frameon=False,
    show=False,
    ax=ax
)

ax.set_title(
    "Diffusion Map of Frozen Malignant States"
)

plt.tight_layout()

fig.savefig(
    FIGURE_DIR /
    "Step27C_Diffmap_DC1_DC2.png",
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    FIGURE_DIR /
    "Step27C_Diffmap_DC1_DC2.pdf",
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
# =============================================================================
# Step27 Cell 16
# Plot DC1 vs DC3
# =============================================================================

fig, ax = plt.subplots(
    figsize=(7, 6)
)

sc.pl.embedding(
    adata27,
    basis="diffmap",
    color=STATE_COL,
    components="1,3",
    size=12,
    alpha=0.7,
    frameon=False,
    show=False,
    ax=ax
)

ax.set_title(
    "Diffusion Structure of Frozen Malignant States"
)

plt.tight_layout()

fig.savefig(
    FIGURE_DIR /
    "Step27C_Diffmap_DC1_DC3.png",
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    FIGURE_DIR /
    "Step27C_Diffmap_DC1_DC3.pdf",
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
# =============================================================================
# Step27 Cell 17
# REPORT — Diffusion-map figures generated
# =============================================================================

print("=" * 80)
print("STEP27C-2 REPORT")
print("=" * 80)

print("Please review/send:")
print("1. Step27C_Diffmap_DC1_DC2.png")
print("2. Step27C_Diffmap_DC1_DC3.png")
print("3. Cell 14 focus-state DC summary table")

print("\nNo pseudotime has been computed yet.")

In [ ]:
# =============================================================================
# Step27 Cell 18
# Patient-aware diffusion summary
# =============================================================================

patient_focus = (
    adata27.obs[
        adata27.obs[PATIENT_COL].isin(
            ["HCC09", "HCC10"]
        )
        &
        adata27.obs[STATE_COL].isin(
            ["MS6", "MS5", "MS3"]
        )
    ]
)

patient_dc_summary = (
    patient_focus
    .groupby(
        [
            PATIENT_COL,
            STATE_COL
        ],
        observed=True
    )[
        [f"DC{i}" for i in range(1, 7)]
    ]
    .agg(
        ["median", "mean", "std", "count"]
    )
)

display(
    patient_dc_summary.round(4)
)

In [ ]:
# =============================================================================
# Step27 Cell 19
# REPORT — HCC09/HCC10 MS6-MS5-MS3 diffusion positions
# =============================================================================

print("=" * 90)
print("STEP27C PATIENT-AWARE REPORT")
print("=" * 90)

for patient in [
    "HCC09",
    "HCC10"
]:

    print(f"\n===== {patient} =====")

    tmp = (
        adata27.obs.loc[
            (adata27.obs[PATIENT_COL] == patient)
            &
            (
                adata27.obs[STATE_COL].isin(
                    ["MS6", "MS5", "MS3"]
                )
            ),
            [
                STATE_COL,
                "DC1",
                "DC2",
                "DC3",
                "DC4",
                "DC5",
                "DC6"
            ]
        ]
        .groupby(
            STATE_COL,
            observed=True
        )
        .median()
    )

    display(
        tmp.round(4)
    )

print(
    "\nREPORT COMPLETE — "
    "NO ROOT / NO DPT YET"
)

In [ ]:
# =============================================================================
# Step27 Cell 20
# All-state diffusion-coordinate summary
# =============================================================================

dc_cols = [
    "DC2",
    "DC3",
    "DC4",
    "DC5",
    "DC6"
]

all_state_dc = (
    adata27.obs
    .groupby(
        STATE_COL,
        observed=True
    )[dc_cols]
    .median()
)

display(
    all_state_dc.round(4)
)

In [ ]:
# =============================================================================
# Step27 Cell 21
# Define frozen biological programs for trajectory interpretation
# =============================================================================

trajectory_programs = [
    "MS3_conserved_Core50_score",
    "Program__Proliferation_G2M",
    "DeepProgram__Mitotic_Entry",
    "DeepProgram__Spindle_Chromosome_Segregation",
    "Program__DNA_Replication_S",
    "DeepProgram__Replication_Stress",
    "DeepProgram__Homologous_Recombination",
    "DeepProgram__Fanconi_DNA_Repair",
    "DeepProgram__Nucleotide_Synthesis",
    "DeepProgram__Hepatic_Differentiation",
    "DeepProgram__Bile_Acid_Metabolism"
]

missing_programs = [
    x for x in trajectory_programs
    if x not in adata27.obs.columns
]

print("Programs found:",
      len(trajectory_programs) - len(missing_programs))

print("Programs missing:",
      missing_programs)

assert len(missing_programs) == 0

In [ ]:
# =============================================================================
# Step27 Cell 22
# Biological correlations of diffusion components
# =============================================================================

from scipy.stats import spearmanr

records = []

for dc in dc_cols:
    for program in trajectory_programs:

        rho, p = spearmanr(
            adata27.obs[dc],
            adata27.obs[program],
            nan_policy="omit"
        )

        records.append({
            "Diffusion_Component": dc,
            "Program": program,
            "Spearman_rho": rho,
            "P_value": p
        })

dc_program_corr = pd.DataFrame(records)

display(
    dc_program_corr
    .sort_values(
        ["Diffusion_Component", "Spearman_rho"],
        ascending=[True, False]
    )
    .round(4)
)

In [ ]:
# =============================================================================
# Step27 Cell 23
# Focused biological interpretation of DC3
# =============================================================================

dc3_corr = (
    dc_program_corr[
        dc_program_corr[
            "Diffusion_Component"
        ] == "DC3"
    ]
    .sort_values(
        "Spearman_rho",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    dc3_corr.round(4)
)

In [ ]:
# =============================================================================
# Step27 Cell 24
# HCC09-specific DC3 biological correlations
# =============================================================================

hcc09 = adata27.obs[
    adata27.obs[PATIENT_COL] == "HCC09"
]

records = []

for program in trajectory_programs:

    rho, p = spearmanr(
        hcc09["DC3"],
        hcc09[program],
        nan_policy="omit"
    )

    records.append({
        "Program": program,
        "Spearman_rho": rho,
        "P_value": p
    })

hcc09_dc3_corr = (
    pd.DataFrame(records)
    .sort_values(
        "Spearman_rho",
        ascending=False
    )
)

display(
    hcc09_dc3_corr.round(4)
)

In [ ]:
# =============================================================================
# Step27 Cell 25
# HCC10-specific DC3 biological correlations
# =============================================================================

hcc10 = adata27.obs[
    adata27.obs[PATIENT_COL] == "HCC10"
]

records = []

for program in trajectory_programs:

    rho, p = spearmanr(
        hcc10["DC3"],
        hcc10[program],
        nan_policy="omit"
    )

    records.append({
        "Program": program,
        "Spearman_rho": rho,
        "P_value": p
    })

hcc10_dc3_corr = (
    pd.DataFrame(records)
    .sort_values(
        "Spearman_rho",
        ascending=False
    )
)

display(
    hcc10_dc3_corr.round(4)
)

In [ ]:
# =============================================================================
# Step27 Cell 26
# Diffusion map: DC2 vs DC3
# =============================================================================

fig, ax = plt.subplots(
    figsize=(7, 6)
)

sc.pl.embedding(
    adata27,
    basis="diffmap",
    color=STATE_COL,
    components="2,3",
    size=12,
    alpha=0.7,
    frameon=False,
    show=False,
    ax=ax
)

ax.set_title(
    "Diffusion Structure of Frozen Malignant States"
)

plt.tight_layout()

fig.savefig(
    FIGURE_DIR /
    "Step27C_Diffmap_DC2_DC3.png",
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    FIGURE_DIR /
    "Step27C_Diffmap_DC2_DC3.pdf",
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
# =============================================================================
# Step27 Cell 27
# DC3 distribution across frozen malignant states
# =============================================================================

import matplotlib.pyplot as plt

plot_data = [
    adata27.obs.loc[
        adata27.obs[STATE_COL] == state,
        "DC3"
    ].values
    for state in STATE_ORDER
]

fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.boxplot(
    plot_data,
    labels=STATE_ORDER,
    showfliers=False
)

ax.set_xlabel(
    "Malignant state"
)

ax.set_ylabel(
    "Diffusion component 3"
)

ax.set_title(
    "DC3 Distribution Across Frozen Malignant States"
)

plt.tight_layout()

fig.savefig(
    FIGURE_DIR /
    "Step27C_DC3_State_Distribution.png",
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    FIGURE_DIR /
    "Step27C_DC3_State_Distribution.pdf",
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
# =============================================================================
# Step27 Cell 28
# REPORT — Diffusion-axis biological validation
# =============================================================================

print("=" * 90)
print("STEP27C DIFFUSION-AXIS REPORT")
print("=" * 90)

print("\n[1] All-state median diffusion coordinates")
display(
    all_state_dc.round(4)
)

print("\n[2] DC3 biological correlations — GLOBAL")
display(
    dc3_corr[
        [
            "Program",
            "Spearman_rho",
            "P_value"
        ]
    ].round(4)
)

print("\n[3] DC3 biological correlations — HCC09")
display(
    hcc09_dc3_corr.round(4)
)

print("\n[4] DC3 biological correlations — HCC10")
display(
    hcc10_dc3_corr.round(4)
)

print("\n[5] MS6 → MS5 → MS3 median DC3")

for patient in [
    "HCC09",
    "HCC10"
]:

    tmp = (
        adata27.obs[
            (adata27.obs[PATIENT_COL] == patient)
            &
            (
                adata27.obs[STATE_COL].isin(
                    ["MS6", "MS5", "MS3"]
                )
            )
        ]
        .groupby(
            STATE_COL,
            observed=True
        )["DC3"]
        .median()
    )

    print(f"\n{patient}")

    for state in [
        "MS6",
        "MS5",
        "MS3"
    ]:
        print(
            f"{state}: "
            f"{tmp.loc[state]:.5f}"
        )

print(
    "\nFigures to review:"
)

print(
    "Step27C_Diffmap_DC2_DC3.png"
)

print(
    "Step27C_DC3_State_Distribution.png"
)

print(
    "\nNO ROOT / NO DPT / "
    "NO DIRECTIONAL CLAIM YET"
)

In [ ]:
# =============================================================================
# Step27 Cell 29
# Prepare MS6 root-selection framework
# =============================================================================

from scipy import sparse
import numpy as np
import pandas as pd

ROOT_DCS = [
    "DC2",
    "DC3",
    "DC4",
    "DC5",
    "DC6"
]

conn = adata27.obsp[
    connectivities_key
].tocsr()

print("Root-selection coordinates:", ROOT_DCS)
print("Graph shape:", conn.shape)

In [ ]:
# =============================================================================
# Step27 Cell 30
# Representative root-selection function
# =============================================================================

def select_representative_root(
    adata,
    cell_mask,
    dc_cols,
    graph,
    degree_quantile=0.50
):
    idx = np.where(
        np.asarray(cell_mask)
    )[0]

    assert len(idx) > 0

    # Within-subset graph degree
    subgraph = graph[idx][:, idx]

    degree = np.asarray(
        subgraph.sum(axis=1)
    ).ravel()

    degree_cutoff = np.quantile(
        degree,
        degree_quantile
    )

    central_mask = (
        degree >= degree_cutoff
    )

    central_idx = idx[
        central_mask
    ]

    # Diffusion coordinates
    coords = adata.obs.loc[
        adata.obs_names[central_idx],
        dc_cols
    ].values

    # Robust center
    center = np.median(
        coords,
        axis=0
    )

    # Scale dimensions
    scale = np.std(
        coords,
        axis=0
    )

    scale[
        scale == 0
    ] = 1

    distance = np.sqrt(
        np.sum(
            (
                (coords - center)
                / scale
            ) ** 2,
            axis=1
        )
    )

    best_local = np.argmin(
        distance
    )

    best_idx = int(
        central_idx[best_local]
    )

    return {
        "root_index": best_idx,
        "root_cell": str(
            adata.obs_names[
                best_idx
            ]
        ),
        "subset_n": int(
            len(idx)
        ),
        "central_candidate_n": int(
            len(central_idx)
        ),
        "degree": float(
            degree[
                np.where(
                    idx == best_idx
                )[0][0]
            ]
        ),
        "distance_to_center": float(
            distance[
                best_local
            ]
        )
    }

In [ ]:
# =============================================================================
# Step27 Cell 31
# Select global / HCC09 / HCC10 MS6 roots
# =============================================================================

ms6_mask = (
    adata27.obs[STATE_COL]
    == "MS6"
)

hcc09_ms6_mask = (
    (adata27.obs[PATIENT_COL] == "HCC09")
    &
    (adata27.obs[STATE_COL] == "MS6")
)

hcc10_ms6_mask = (
    (adata27.obs[PATIENT_COL] == "HCC10")
    &
    (adata27.obs[STATE_COL] == "MS6")
)

root_global = select_representative_root(
    adata27,
    ms6_mask,
    ROOT_DCS,
    conn
)

root_hcc09 = select_representative_root(
    adata27,
    hcc09_ms6_mask,
    ROOT_DCS,
    conn
)

root_hcc10 = select_representative_root(
    adata27,
    hcc10_ms6_mask,
    ROOT_DCS,
    conn
)

root_table = pd.DataFrame({
    "Root": [
        "Global_MS6",
        "HCC09_MS6",
        "HCC10_MS6"
    ],
    "Cell": [
        root_global["root_cell"],
        root_hcc09["root_cell"],
        root_hcc10["root_cell"]
    ],
    "Index": [
        root_global["root_index"],
        root_hcc09["root_index"],
        root_hcc10["root_index"]
    ],
    "Subset_N": [
        root_global["subset_n"],
        root_hcc09["subset_n"],
        root_hcc10["subset_n"]
    ],
    "Central_candidate_N": [
        root_global["central_candidate_n"],
        root_hcc09["central_candidate_n"],
        root_hcc10["central_candidate_n"]
    ]
})

display(root_table)

In [ ]:
# =============================================================================
# Step27 Cell 32
# Primary MS6-anchored DPT
# =============================================================================

adata27.uns["iroot"] = (
    root_global["root_index"]
)

sc.tl.dpt(
    adata27,
    n_dcs=10,
    neighbors_key=NEIGHBOR_KEY
)

adata27.obs[
    "dpt_pseudotime_globalMS6"
] = adata27.obs[
    "dpt_pseudotime"
].copy()

print(
    adata27.obs[
        "dpt_pseudotime_globalMS6"
    ].describe()
)

print(
    "\nFinite fraction:",
    np.isfinite(
        adata27.obs[
            "dpt_pseudotime_globalMS6"
        ]
    ).mean()
)

In [ ]:
# =============================================================================
# Step27 Cell 33
# Alternative-root DPT sensitivity
# =============================================================================

# HCC09 MS6 root
adata27.uns["iroot"] = (
    root_hcc09["root_index"]
)

sc.tl.dpt(
    adata27,
    n_dcs=10,
    neighbors_key=NEIGHBOR_KEY
)

adata27.obs[
    "dpt_pseudotime_HCC09MS6"
] = adata27.obs[
    "dpt_pseudotime"
].copy()


# HCC10 MS6 root
adata27.uns["iroot"] = (
    root_hcc10["root_index"]
)

sc.tl.dpt(
    adata27,
    n_dcs=10,
    neighbors_key=NEIGHBOR_KEY
)

adata27.obs[
    "dpt_pseudotime_HCC10MS6"
] = adata27.obs[
    "dpt_pseudotime"
].copy()

print("Alternative-root DPT completed.")

In [ ]:
# =============================================================================
# Step27 Cell 34
# Root-sensitivity correlation
# =============================================================================

from scipy.stats import spearmanr

PT_COLS = [
    "dpt_pseudotime_globalMS6",
    "dpt_pseudotime_HCC09MS6",
    "dpt_pseudotime_HCC10MS6"
]

pt_corr = pd.DataFrame(
    index=PT_COLS,
    columns=PT_COLS,
    dtype=float
)

for x in PT_COLS:
    for y in PT_COLS:

        valid = (
            np.isfinite(
                adata27.obs[x]
            )
            &
            np.isfinite(
                adata27.obs[y]
            )
        )

        rho, _ = spearmanr(
            adata27.obs.loc[
                valid, x
            ],
            adata27.obs.loc[
                valid, y
            ]
        )

        pt_corr.loc[x, y] = rho

display(
    pt_corr.round(4)
)

In [ ]:
# =============================================================================
# Step27 Cell 35
# State-level pseudotime ordering
# =============================================================================

state_pt_median = (
    adata27.obs
    .groupby(
        STATE_COL,
        observed=True
    )[PT_COLS]
    .median()
)

display(
    state_pt_median.round(4)
)

In [ ]:
# =============================================================================
# Step27 Cell 36
# Within-patient MS6-MS5-MS3 pseudotime
# =============================================================================

focus_patient_pt = (
    adata27.obs[
        adata27.obs[PATIENT_COL].isin(
            ["HCC09", "HCC10"]
        )
        &
        adata27.obs[STATE_COL].isin(
            ["MS6", "MS5", "MS3"]
        )
    ]
    .groupby(
        [
            PATIENT_COL,
            STATE_COL
        ],
        observed=True
    )[PT_COLS]
    .median()
)

display(
    focus_patient_pt.round(4)
)

In [ ]:
# =============================================================================
# Step27 Cell 37
# REPORT — MS6-root DPT robustness
# =============================================================================

print("=" * 90)
print("STEP27 DPT ROOT-SENSITIVITY REPORT")
print("=" * 90)

print("\n[1] Selected roots")
display(
    root_table
)

print("\n[2] Pseudotime root-to-root Spearman correlation")
display(
    pt_corr.round(4)
)

print("\n[3] State-level median pseudotime")
display(
    state_pt_median.round(4)
)

print("\n[4] HCC09/HCC10 MS6-MS5-MS3 median pseudotime")
display(
    focus_patient_pt.round(4)
)

print("\n[5] Primary root")

print(
    root_global["root_cell"],
    "index =",
    root_global["root_index"]
)

print(
    "\nNo lineage-conversion claim is made."
)

print(
    "DPT is interpreted as "
    "MS6-anchored pseudotemporal ordering."
)

In [ ]:
# =============================================================================
# Step27 Cell 38
# Lock primary MS6-anchored pseudotime
# =============================================================================

PRIMARY_PT = "dpt_pseudotime_globalMS6"

adata27.obs[
    "MS6_anchored_pseudotime"
] = adata27.obs[PRIMARY_PT].copy()

print("Primary pseudotime:", PRIMARY_PT)

print(
    adata27.obs[
        "MS6_anchored_pseudotime"
    ].describe()
)

In [ ]:
# =============================================================================
# Step27 Cell 39
# Define primary trajectory-analysis cells
# =============================================================================

trajectory_mask = (
    adata27.obs[PATIENT_COL].isin(
        ["HCC09", "HCC10"]
    )
    &
    adata27.obs[STATE_COL].isin(
        ["MS6", "MS5", "MS3"]
    )
)

traj_obs = adata27.obs.loc[
    trajectory_mask
].copy()

print("Primary trajectory cells:", len(traj_obs))

display(
    pd.crosstab(
        traj_obs[PATIENT_COL],
        traj_obs[STATE_COL]
    )
)

In [ ]:
# =============================================================================
# Step27 Cell 40
# Define biological programs for pseudotime analysis
# =============================================================================

primary_programs = [
    "MS3_conserved_Core50_score",
    "Program__Proliferation_G2M",
    "DeepProgram__Mitotic_Entry",
    "DeepProgram__Spindle_Chromosome_Segregation",
    "DeepProgram__Hepatic_Differentiation",
    "DeepProgram__Bile_Acid_Metabolism"
]

supporting_programs = [
    "Program__DNA_Replication_S",
    "DeepProgram__Replication_Stress",
    "DeepProgram__Homologous_Recombination",
    "DeepProgram__Fanconi_DNA_Repair",
    "DeepProgram__Nucleotide_Synthesis"
]

step27d_programs = (
    primary_programs
    + supporting_programs
)

missing = [
    x for x in step27d_programs
    if x not in traj_obs.columns
]

print("Missing programs:", missing)

assert len(missing) == 0

In [ ]:
# =============================================================================
# Step27 Cell 41
# Patient-specific program-pseudotime correlations
# =============================================================================

from scipy.stats import spearmanr

records = []

for patient in ["HCC09", "HCC10"]:

    tmp = traj_obs[
        traj_obs[PATIENT_COL] == patient
    ]

    for program in step27d_programs:

        rho, p = spearmanr(
            tmp["MS6_anchored_pseudotime"],
            tmp[program],
            nan_policy="omit"
        )

        records.append({
            "Patient": patient,
            "Program": program,
            "Spearman_rho": rho,
            "P_value": p,
            "N_cells": len(tmp)
        })

patient_pt_corr = pd.DataFrame(
    records
)

display(
    patient_pt_corr.round(4)
)

In [ ]:
# =============================================================================
# Step27 Cell 42
# Cross-patient concordance
# =============================================================================

corr_wide = (
    patient_pt_corr
    .pivot(
        index="Program",
        columns="Patient",
        values="Spearman_rho"
    )
)

corr_wide["Same_direction"] = (
    np.sign(corr_wide["HCC09"])
    ==
    np.sign(corr_wide["HCC10"])
)

corr_wide["Mean_rho"] = (
    corr_wide[
        ["HCC09", "HCC10"]
    ].mean(axis=1)
)

corr_wide = corr_wide.sort_values(
    "Mean_rho",
    ascending=False
)

display(
    corr_wide.round(4)
)

In [ ]:
# =============================================================================
# Step27 Cell 43
# REPORT — Patient-aware pseudotime biology
# =============================================================================

print("=" * 90)
print("STEP27D-1 REPORT")
print("=" * 90)

print(
    "\nProgram correlations with "
    "MS6-anchored pseudotime:"
)

display(
    corr_wide.round(4)
)

print(
    "\nConcordant programs:",
    int(
        corr_wide[
            "Same_direction"
        ].sum()
    ),
    "/",
    len(corr_wide)
)

In [ ]:
# =============================================================================
# Step27 Cell 44
# Patient-specific pseudotime bins
# =============================================================================

traj_obs[
    "PT_bin"
] = np.nan

for patient in [
    "HCC09",
    "HCC10"
]:

    mask = (
        traj_obs[PATIENT_COL]
        == patient
    )

    traj_obs.loc[
        mask,
        "PT_bin"
    ] = pd.qcut(
        traj_obs.loc[
            mask,
            "MS6_anchored_pseudotime"
        ],
        q=5,
        labels=[
            "Q1",
            "Q2",
            "Q3",
            "Q4",
            "Q5"
        ],
        duplicates="drop"
    ).astype(str)

print(
    pd.crosstab(
        traj_obs[PATIENT_COL],
        traj_obs["PT_bin"]
    )
)

In [ ]:
# =============================================================================
# Step27 Cell 45
# Patient-wise standardization for visualization
# =============================================================================

for program in step27d_programs:

    z_col = program + "__z"

    traj_obs[z_col] = (
        traj_obs
        .groupby(
            PATIENT_COL
        )[program]
        .transform(
            lambda x:
            (x - x.mean())
            / x.std()
        )
    )

print("Patient-wise z-scores created.")

In [ ]:
# =============================================================================
# Step27 Cell 46
# Program dynamics across pseudotime bins
# =============================================================================

z_programs = [
    x + "__z"
    for x in step27d_programs
]

pt_bin_summary = (
    traj_obs
    .groupby(
        [
            PATIENT_COL,
            "PT_bin"
        ],
        observed=True
    )[z_programs]
    .median()
    .reset_index()
)

display(
    pt_bin_summary.round(3)
)

In [ ]:
# =============================================================================
# Step27 Cell 47
# Patient-balanced pseudotime trends
# =============================================================================

balanced_pt_summary = (
    pt_bin_summary
    .groupby(
        "PT_bin",
        observed=True
    )[z_programs]
    .mean()
    .reindex(
        ["Q1", "Q2", "Q3", "Q4", "Q5"]
    )
)

display(
    balanced_pt_summary.round(3)
)

In [ ]:
# =============================================================================
# Step27 Cell 48
# Plot primary biological dynamics
# =============================================================================

primary_z = [
    x + "__z"
    for x in primary_programs
]

fig, ax = plt.subplots(
    figsize=(9, 6)
)

x = np.arange(1, 6)

for program in primary_z:

    label = (
        program
        .replace("__z", "")
        .replace("DeepProgram__", "")
        .replace("Program__", "")
    )

    ax.plot(
        x,
        balanced_pt_summary[
            program
        ].values,
        marker="o",
        label=label
    )

ax.axhline(
    0,
    linewidth=1
)

ax.set_xticks(
    x
)

ax.set_xticklabels(
    [
        "Early",
        "Q2",
        "Q3",
        "Q4",
        "Late"
    ]
)

ax.set_xlabel(
    "MS6-anchored pseudotime"
)

ax.set_ylabel(
    "Patient-standardized program score"
)

ax.set_title(
    "Biological Dynamics Along MS6-Anchored Pseudotime"
)

ax.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False
)

plt.tight_layout()

fig.savefig(
    FIGURE_DIR /
    "Step27D_Primary_Program_Dynamics.png",
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    FIGURE_DIR /
    "Step27D_Primary_Program_Dynamics.pdf",
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
# =============================================================================
# Step27 Cell 49
# Supporting pathway dynamics
# =============================================================================

supporting_z = [
    x + "__z"
    for x in supporting_programs
]

fig, ax = plt.subplots(
    figsize=(9, 6)
)

x = np.arange(1, 6)

for program in supporting_z:

    label = (
        program
        .replace("__z", "")
        .replace("DeepProgram__", "")
        .replace("Program__", "")
    )

    ax.plot(
        x,
        balanced_pt_summary[
            program
        ].values,
        marker="o",
        label=label
    )

ax.axhline(
    0,
    linewidth=1
)

ax.set_xticks(x)

ax.set_xticklabels(
    [
        "Early",
        "Q2",
        "Q3",
        "Q4",
        "Late"
    ]
)

ax.set_xlabel(
    "MS6-anchored pseudotime"
)

ax.set_ylabel(
    "Patient-standardized program score"
)

ax.set_title(
    "Supporting Programs Along Pseudotime"
)

ax.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False
)

plt.tight_layout()

fig.savefig(
    FIGURE_DIR /
    "Step27D_Supporting_Program_Dynamics.png",
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    FIGURE_DIR /
    "Step27D_Supporting_Program_Dynamics.pdf",
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
# =============================================================================
# Step27 Cell 50
# State composition across pseudotime bins
# =============================================================================

state_bin_counts = pd.crosstab(
    traj_obs["PT_bin"],
    traj_obs[STATE_COL]
)

state_bin_fraction = (
    state_bin_counts
    .div(
        state_bin_counts.sum(
            axis=1
        ),
        axis=0
    )
    .reindex(
        ["Q1", "Q2", "Q3", "Q4", "Q5"]
    )
)

display(
    state_bin_fraction.round(3)
)

In [ ]:
# =============================================================================
# Step27 Cell 51
# Plot state composition across pseudotime
# =============================================================================

ax = state_bin_fraction[
    ["MS6", "MS5", "MS3"]
].plot(
    kind="bar",
    stacked=True,
    figsize=(8, 5)
)

ax.set_xlabel(
    "MS6-anchored pseudotime bin"
)

ax.set_ylabel(
    "Cell fraction"
)

ax.set_title(
    "Malignant-State Composition Along Pseudotime"
)

ax.legend(
    title="State",
    frameon=False
)

plt.tight_layout()

fig = ax.get_figure()

fig.savefig(
    FIGURE_DIR /
    "Step27D_StateComposition_Pseudotime.png",
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    FIGURE_DIR /
    "Step27D_StateComposition_Pseudotime.pdf",
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
# =============================================================================
# Step27 Cell 52
# REPORT — Biological dynamics along pseudotime
# =============================================================================

print("=" * 100)
print("STEP27D FINAL REPORT")
print("=" * 100)

print(
    "\n[1] Patient-specific "
    "program-pseudotime correlations"
)

display(
    corr_wide.round(4)
)

print(
    "\n[2] Patient-balanced "
    "pseudotime-bin program dynamics"
)

display(
    balanced_pt_summary[
        primary_z
    ].round(3)
)

print(
    "\n[3] Supporting program dynamics"
)

display(
    balanced_pt_summary[
        supporting_z
    ].round(3)
)

print(
    "\n[4] State composition "
    "across pseudotime"
)

display(
    state_bin_fraction[
        ["MS6", "MS5", "MS3"]
    ].round(3)
)

print(
    "\nFigures:"
)

print(
    "Step27D_Primary_Program_Dynamics.png"
)

print(
    "Step27D_Supporting_Program_Dynamics.png"
)

print(
    "Step27D_StateComposition_Pseudotime.png"
)

print(
    "\nInterpretation boundary:"
)

print(
    "These results describe "
    "MS6-anchored pseudotemporal compatibility "
    "and biological program dynamics."
)

print(
    "They do not demonstrate actual "
    "lineage conversion from MS6 to MS3."
)

In [ ]:
# =============================================================================
# Step27 Cell 53
# Pseudotime distribution across all frozen malignant states
# =============================================================================

state_pt_summary = (
    adata27.obs
    .groupby(
        STATE_COL,
        observed=True
    )["MS6_anchored_pseudotime"]
    .agg([
        "count",
        "median",
        lambda x: x.quantile(0.25),
        lambda x: x.quantile(0.75)
    ])
)

state_pt_summary.columns = [
    "N",
    "Median_PT",
    "Q25_PT",
    "Q75_PT"
]

display(
    state_pt_summary.round(4)
)

In [ ]:
# =============================================================================
# Step27 Cell 54
# Patient support of each malignant state
# =============================================================================

patient_state_support = pd.crosstab(
    adata27.obs[STATE_COL],
    adata27.obs[PATIENT_COL]
)

patient_state_support[
    "N_patients_with_cells"
] = (
    patient_state_support > 0
).sum(axis=1)

display(
    patient_state_support
)

In [ ]:
# =============================================================================
# Step27 Cell 55
# Quantify intermediate positioning of MS5
# =============================================================================

intermediate_records = []

for patient in [
    "HCC09",
    "HCC10"
]:

    tmp = (
        adata27.obs[
            (adata27.obs[PATIENT_COL] == patient)
            &
            adata27.obs[STATE_COL].isin(
                ["MS6", "MS5", "MS3"]
            )
        ]
        .groupby(
            STATE_COL,
            observed=True
        )["MS6_anchored_pseudotime"]
        .median()
    )

    ms6 = tmp.loc["MS6"]
    ms5 = tmp.loc["MS5"]
    ms3 = tmp.loc["MS3"]

    position = (
        (ms5 - ms6)
        /
        (ms3 - ms6)
    )

    intermediate_records.append({
        "Patient": patient,
        "MS6_median_PT": ms6,
        "MS5_median_PT": ms5,
        "MS3_median_PT": ms3,
        "MS5_relative_position": position
    })

ms5_position = pd.DataFrame(
    intermediate_records
)

display(
    ms5_position.round(4)
)

In [ ]:
# =============================================================================
# Step27 Cell 56
# HCC10 positioning of MS1 relative to MS6-MS5-MS3
# =============================================================================

hcc10_state_pt = (
    adata27.obs[
        (adata27.obs[PATIENT_COL] == "HCC10")
        &
        adata27.obs[STATE_COL].isin(
            ["MS6", "MS5", "MS3", "MS1"]
        )
    ]
    .groupby(
        STATE_COL,
        observed=True
    )["MS6_anchored_pseudotime"]
    .agg([
        "count",
        "median",
        "mean"
    ])
)

display(
    hcc10_state_pt.round(4)
)

In [ ]:
# =============================================================================
# Step27 Cell 57
# HCC10 biological profiles of MS6/MS5/MS3/MS1
# =============================================================================

hcc10_focus = adata27.obs[
    (adata27.obs[PATIENT_COL] == "HCC10")
    &
    adata27.obs[STATE_COL].isin(
        ["MS6", "MS5", "MS3", "MS1"]
    )
]

hcc10_program_summary = (
    hcc10_focus
    .groupby(
        STATE_COL,
        observed=True
    )[primary_programs]
    .median()
)

display(
    hcc10_program_summary.round(3)
)

In [ ]:
# =============================================================================
# Step27 Cell 58
# Pseudotime distribution across all malignant states
# =============================================================================

plot_data = [
    adata27.obs.loc[
        adata27.obs[STATE_COL] == state,
        "MS6_anchored_pseudotime"
    ].values
    for state in STATE_ORDER
]

fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.boxplot(
    plot_data,
    tick_labels=STATE_ORDER,
    showfliers=False
)

ax.set_xlabel(
    "Malignant state"
)

ax.set_ylabel(
    "MS6-anchored pseudotime"
)

ax.set_title(
    "Pseudotemporal Position of Frozen Malignant States"
)

plt.tight_layout()

fig.savefig(
    FIGURE_DIR /
    "Step27E_State_Pseudotime_Distribution.png",
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    FIGURE_DIR /
    "Step27E_State_Pseudotime_Distribution.pdf",
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
# =============================================================================
# Step27 Cell 59
# REPORT — MS1 / MS5 positioning
# =============================================================================

print("=" * 95)
print("STEP27E REPORT — MS1 / MS5 POSITIONING")
print("=" * 95)

print(
    "\n[1] All-state pseudotime summary"
)

display(
    state_pt_summary.round(4)
)

print(
    "\n[2] Patient support"
)

display(
    patient_state_support
)

print(
    "\n[3] MS5 relative position "
    "between MS6 and MS3"
)

display(
    ms5_position.round(4)
)

print(
    "\n[4] HCC10 MS1/MS5/MS6/MS3 pseudotime"
)

display(
    hcc10_state_pt.round(4)
)

print(
    "\n[5] HCC10 biological profiles"
)

display(
    hcc10_program_summary.round(3)
)

print(
    "\nFigure:"
)

print(
    "Step27E_State_Pseudotime_Distribution.png"
)

print(
    "\nInterpretation remains topology-aware:"
)

print(
    "MS5 may show intermediate-like positioning, "
    "whereas MS1 may represent a branch-associated state. "
    "No lineage conversion is inferred."
)

In [ ]:
# =============================================================================
# Step27 Cell 60
# Patient-aware MS6-MS5-MS3 ordering robustness
# =============================================================================

records = []

for patient in ["HCC09", "HCC10"]:

    tmp = (
        adata27.obs[
            (adata27.obs[PATIENT_COL] == patient)
            &
            adata27.obs[STATE_COL].isin(
                ["MS6", "MS5", "MS3"]
            )
        ]
        .groupby(
            STATE_COL,
            observed=True
        )["MS6_anchored_pseudotime"]
        .median()
    )

    ms6 = float(tmp.loc["MS6"])
    ms5 = float(tmp.loc["MS5"])
    ms3 = float(tmp.loc["MS3"])

    records.append({
        "Patient": patient,
        "MS6_median_PT": ms6,
        "MS5_median_PT": ms5,
        "MS3_median_PT": ms3,
        "MS5_minus_MS6": ms5 - ms6,
        "MS3_minus_MS5": ms3 - ms5,
        "MS3_minus_MS6": ms3 - ms6,
        "Monotonic_MS6_MS5_MS3": (
            ms6 < ms5 < ms3
        )
    })

patient_axis_robustness = pd.DataFrame(records)

display(
    patient_axis_robustness.round(4)
)

In [ ]:
# =============================================================================
# Step27 Cell 61
# Root sensitivity summary
# =============================================================================

root_sensitivity_summary = pd.DataFrame({
    "Comparison": [
        "Global vs HCC09 root",
        "Global vs HCC10 root",
        "HCC09 vs HCC10 root"
    ],
    "Spearman_rho": [
        pt_corr.loc[
            "dpt_pseudotime_globalMS6",
            "dpt_pseudotime_HCC09MS6"
        ],
        pt_corr.loc[
            "dpt_pseudotime_globalMS6",
            "dpt_pseudotime_HCC10MS6"
        ],
        pt_corr.loc[
            "dpt_pseudotime_HCC09MS6",
            "dpt_pseudotime_HCC10MS6"
        ]
    ]
})

display(
    root_sensitivity_summary.round(4)
)

In [ ]:
# =============================================================================
# Step27 Cell 62
# Cross-patient biological concordance
# =============================================================================

program_robustness = corr_wide.copy()

program_robustness["Program"] = (
    program_robustness.index
)

program_robustness["Program_Class"] = np.where(
    program_robustness["Program"].isin(
        primary_programs
    ),
    "Primary",
    "Supporting"
)

program_robustness = (
    program_robustness.reset_index(drop=True)
)

display(
    program_robustness[
        [
            "Program",
            "Program_Class",
            "HCC09",
            "HCC10",
            "Same_direction",
            "Mean_rho"
        ]
    ].round(4)
)

In [ ]:
# =============================================================================
# Step27 Cell 63
# Patient-specific program correlation heatmap
# =============================================================================

heatmap_data = (
    program_robustness
    .set_index("Program")[
        ["HCC09", "HCC10"]
    ]
)

fig, ax = plt.subplots(
    figsize=(7, 7)
)

im = ax.imshow(
    heatmap_data.values,
    aspect="auto"
)

ax.set_xticks(
    [0, 1]
)

ax.set_xticklabels(
    ["HCC09", "HCC10"]
)

ax.set_yticks(
    range(len(heatmap_data))
)

ax.set_yticklabels(
    [
        x.replace("DeepProgram__", "")
         .replace("Program__", "")
        for x in heatmap_data.index
    ]
)

for i in range(
    heatmap_data.shape[0]
):
    for j in range(
        heatmap_data.shape[1]
    ):

        ax.text(
            j,
            i,
            f"{heatmap_data.iloc[i, j]:.2f}",
            ha="center",
            va="center",
            fontsize=8
        )

cbar = fig.colorbar(
    im,
    ax=ax
)

cbar.set_label(
    "Spearman correlation"
)

ax.set_title(
    "Patient-Aware Program Dynamics"
)

plt.tight_layout()

fig.savefig(
    FIGURE_DIR /
    "Step27F_PatientAware_ProgramCorrelation.png",
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    FIGURE_DIR /
    "Step27F_PatientAware_ProgramCorrelation.pdf",
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
# =============================================================================
# Step27 Cell 64
# Patient-specific pseudotime ordering
# =============================================================================

fig, ax = plt.subplots(
    figsize=(7, 5)
)

x = [0, 1, 2]

for _, row in (
    patient_axis_robustness.iterrows()
):

    ax.plot(
        x,
        [
            row["MS6_median_PT"],
            row["MS5_median_PT"],
            row["MS3_median_PT"]
        ],
        marker="o",
        label=row["Patient"]
    )

ax.set_xticks(x)

ax.set_xticklabels(
    ["MS6", "MS5", "MS3"]
)

ax.set_ylabel(
    "Median MS6-anchored pseudotime"
)

ax.set_xlabel(
    "Malignant state"
)

ax.set_title(
    "Cross-Patient MS6–MS5–MS3 Ordering"
)

ax.legend(
    frameon=False
)

plt.tight_layout()

fig.savefig(
    FIGURE_DIR /
    "Step27F_CrossPatient_StateOrdering.png",
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    FIGURE_DIR /
    "Step27F_CrossPatient_StateOrdering.pdf",
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
# =============================================================================
# Step27 Cell 65
# REPORT — Step27F patient-aware robustness
# =============================================================================

print("=" * 100)
print("STEP27F FINAL ROBUSTNESS REPORT")
print("=" * 100)

print("\n[1] MS6-MS5-MS3 ordering")
display(
    patient_axis_robustness.round(4)
)

print("\n[2] Root sensitivity")
display(
    root_sensitivity_summary.round(4)
)

print("\n[3] Cross-patient biological concordance")
display(
    program_robustness[
        [
            "Program",
            "Program_Class",
            "HCC09",
            "HCC10",
            "Same_direction",
            "Mean_rho"
        ]
    ].round(4)
)

print(
    "\nMonotonic ordering preserved in all patients:",
    patient_axis_robustness[
        "Monotonic_MS6_MS5_MS3"
    ].all()
)

print(
    "Biological programs with concordant direction:",
    int(
        program_robustness[
            "Same_direction"
        ].sum()
    ),
    "/",
    len(program_robustness)
)

print(
    "\nSTEP27F COMPLETE — "
    "PATIENT-AWARE ROBUSTNESS VERIFIED"
)

In [ ]:
# =============================================================================
# Step27 Cell 66
# Final-lock output paths
# =============================================================================

FINAL_DATA_DIR = (
    STEP27_DIR / "Data"
)

FINAL_TABLE_DIR = (
    STEP27_DIR / "Tables"
)

FINAL_MANIFEST_DIR = (
    STEP27_DIR / "Manifest"
)

for d in [
    FINAL_DATA_DIR,
    FINAL_TABLE_DIR,
    FINAL_MANIFEST_DIR
]:
    d.mkdir(
        parents=True,
        exist_ok=True
    )

FINAL_H5AD = (
    FINAL_DATA_DIR /
    "GSE149614_STEP27_FINAL_TRAJECTORY.h5ad"
)

print("Final Step27 h5ad:")
print(FINAL_H5AD)

In [ ]:
# =============================================================================
# Step27 Cell 67
# Restore canonical primary root and pseudotime
# =============================================================================

adata27.uns[
    "iroot"
] = root_global["root_index"]

adata27.obs[
    "dpt_pseudotime"
] = adata27.obs[
    "dpt_pseudotime_globalMS6"
].copy()

adata27.obs[
    "MS6_anchored_pseudotime"
] = adata27.obs[
    "dpt_pseudotime_globalMS6"
].copy()

print(
    "Primary root restored:",
    root_global["root_cell"]
)

print(
    "Primary pseudotime restored."
)

In [ ]:
# =============================================================================
# Step27 Cell 68
# Save cell-level trajectory data
# =============================================================================

cell_level_cols = [
    PATIENT_COL,
    STATE_COL,
    "MS6_anchored_pseudotime",
    "dpt_pseudotime_globalMS6",
    "dpt_pseudotime_HCC09MS6",
    "dpt_pseudotime_HCC10MS6",
    "DC2",
    "DC3",
    "DC4",
    "DC5",
    "DC6",
    "MS3_conserved_Core50_score"
]

cell_trajectory = (
    adata27.obs[
        cell_level_cols
    ]
    .copy()
)

cell_trajectory.insert(
    0,
    "Cell_ID",
    cell_trajectory.index
)

cell_trajectory.to_csv(
    FINAL_TABLE_DIR /
    "Step27_Final_CellLevel_Pseudotime.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "Cell-level trajectory table saved:",
    cell_trajectory.shape
)

In [ ]:
# =============================================================================
# Step27 Cell 69
# Save final Step27 summary tables
# =============================================================================

state_pt_summary.to_csv(
    FINAL_TABLE_DIR /
    "Step27_Final_State_Pseudotime_Summary.csv",
    encoding="utf-8-sig"
)

patient_axis_robustness.to_csv(
    FINAL_TABLE_DIR /
    "Step27_Final_PatientAxis_Robustness.csv",
    index=False,
    encoding="utf-8-sig"
)

root_sensitivity_summary.to_csv(
    FINAL_TABLE_DIR /
    "Step27_Final_RootSensitivity.csv",
    index=False,
    encoding="utf-8-sig"
)

program_robustness.to_csv(
    FINAL_TABLE_DIR /
    "Step27_Final_Program_Robustness.csv",
    index=False,
    encoding="utf-8-sig"
)

balanced_pt_summary.to_csv(
    FINAL_TABLE_DIR /
    "Step27_Final_ProgramDynamics.csv",
    encoding="utf-8-sig"
)

state_bin_fraction.to_csv(
    FINAL_TABLE_DIR /
    "Step27_Final_StateComposition_Pseudotime.csv",
    encoding="utf-8-sig"
)

paga_conn_df.to_csv(
    FINAL_TABLE_DIR /
    "Step27_Final_PAGA_Connectivity.csv",
    encoding="utf-8-sig"
)

print(
    "Final Step27 summary tables saved."
)

In [ ]:
# =============================================================================
# Step27 Cell 70
# Save final Step27 AnnData
# =============================================================================

assert (
    str(FINAL_H5AD)
    !=
    str(INPUT_H5AD)
)

adata27.write_h5ad(
    FINAL_H5AD
)

print(
    "Saved:",
    FINAL_H5AD
)

print(
    "Step26 frozen source remains unchanged."
)

In [ ]:
# =============================================================================
# Step27 Cell 71
# Create Step27 Final Lock
# =============================================================================

step27_lock = {
    "Step": "Step27",
    "Status": "FROZEN",

    "Input": str(
        INPUT_H5AD
    ),

    "Final_H5AD": str(
        FINAL_H5AD
    ),

    "N_cells": int(
        adata27.n_obs
    ),

    "N_genes": int(
        adata27.n_vars
    ),

    "Frozen_state_column":
        STATE_COL,

    "Patient_column":
        PATIENT_COL,

    "Harmony_representation":
        HARMONY_KEY,

    "Neighbor_graph":
        NEIGHBOR_KEY,

    "Primary_root": {
        "Cell": root_global[
            "root_cell"
        ],
        "Index": int(
            root_global[
                "root_index"
            ]
        ),
        "State": "MS6"
    },

    "Primary_pseudotime":
        "MS6_anchored_pseudotime",

    "Root_sensitivity":
        root_sensitivity_summary
        .to_dict(
            orient="records"
        ),

    "Primary_axis":
        "MS6-MS5-MS3",

    "Primary_interpretation":
        (
            "MS6 Hepatic-Differentiated "
            "to MS5 Stress-Inflammatory-Progenitor "
            "to MS3 Cycling-G2M forms a reproducible "
            "pseudotemporal-compatible malignant-state axis."
        ),

    "Biological_dynamics": {
        "Increasing": [
            "MS3 Core50",
            "Proliferation/G2M",
            "Mitotic entry",
            "Spindle/chromosome segregation"
        ],
        "Decreasing": [
            "Hepatic differentiation",
            "Bile-acid metabolism"
        ],
        "Supporting_increase": [
            "Replication stress",
            "Homologous recombination",
            "Fanconi DNA repair",
            "Nucleotide synthesis"
        ],
        "Non_concordant": [
            "DNA replication S-phase"
        ]
    },

    "MS5_interpretation":
        (
            "Intermediate-like / transitional-like "
            "positioning along the MS6-MS3 axis."
        ),

    "MS1_interpretation":
        (
            "Branch-associated hypoxia/TGFb/"
            "inflammatory state rather than "
            "a linear extension of MS3."
        ),

    "Patient_specific_branches": [
        "MS4: HCC08-dominant",
        "MS7/MS8: HCC04-dominant"
    ],

    "Claim_boundary":
        (
            "Step27 demonstrates graph continuity, "
            "diffusion structure and MS6-anchored "
            "pseudotemporal compatibility. "
            "It does not demonstrate actual temporal "
            "evolution or lineage conversion."
        ),

    "RNA_velocity_input":
        "NOT_PRESENT",

    "Frozen_constraints": [
        "No reclustering",
        "No malignant-state redefinition",
        "No Core50 modification",
        "No Original RiskScore reconstruction"
    ]
}

LOCK_PATH = (
    FINAL_MANIFEST_DIR /
    "Step27_FINAL_Lock.json"
)

with open(
    LOCK_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        step27_lock,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    "Saved:",
    LOCK_PATH
)

In [ ]:
# =============================================================================
# Step27 Cell 72
# Output manifest
# =============================================================================

step27_files = []

for folder in [
    TABLE_DIR,
    FIGURE_DIR,
    DATA_DIR,
    MANIFEST_DIR
]:

    for path in folder.rglob("*"):

        if path.is_file():

            step27_files.append({
                "File": path.name,
                "Full_Path": str(path),
                "Extension": path.suffix,
                "Size_MB": round(
                    path.stat().st_size
                    / 1024**2,
                    3
                )
            })

step27_manifest = pd.DataFrame(
    step27_files
)

step27_manifest.to_csv(
    FINAL_MANIFEST_DIR /
    "Step27_FINAL_Output_Manifest.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "Manifest files:",
    len(step27_manifest)
)

display(
    step27_manifest.tail(20)
)

In [ ]:
# =============================================================================
# Step27 Cell 73
# Final integrity check
# =============================================================================

checks = {
    "N_cells_12057":
        adata27.n_obs == 12057,

    "N_states_8":
        adata27.obs[
            STATE_COL
        ].nunique() == 8,

    "Primary_PT_present":
        "MS6_anchored_pseudotime"
        in adata27.obs.columns,

    "Global_PT_present":
        "dpt_pseudotime_globalMS6"
        in adata27.obs.columns,

    "HCC09_PT_present":
        "dpt_pseudotime_HCC09MS6"
        in adata27.obs.columns,

    "HCC10_PT_present":
        "dpt_pseudotime_HCC10MS6"
        in adata27.obs.columns,

    "Final_h5ad_exists":
        FINAL_H5AD.exists(),

    "Final_lock_exists":
        LOCK_PATH.exists(),

    "All_patient_ordering_PASS":
        bool(
            patient_axis_robustness[
                "Monotonic_MS6_MS5_MS3"
            ].all()
        )
}

integrity_df = pd.DataFrame(
    {
        "Check": checks.keys(),
        "PASS": checks.values()
    }
)

display(
    integrity_df
)

assert integrity_df[
    "PASS"
].all()

print(
    "\nALL STEP27 FINAL CHECKS PASSED"
)

In [ ]:
# =============================================================================
# Step27 Cell 74
# FINAL REPORT
# =============================================================================

print("=" * 105)
print("STEP27 FINAL LOCK REPORT")
print("=" * 105)

print(
    "\nFinal object:"
)

print(
    FINAL_H5AD
)

print(
    "\nCells / genes:",
    adata27.n_obs,
    "/",
    adata27.n_vars
)

print(
    "\nPrimary axis:"
)

print(
    "MS6 Hepatic-Differentiated"
    " → "
    "MS5 Stress-Inflammatory-Progenitor"
    " → "
    "MS3 Cycling-G2M"
)

print(
    "\nRoot sensitivity:"
)

display(
    root_sensitivity_summary.round(4)
)

print(
    "\nCross-patient ordering:"
)

display(
    patient_axis_robustness.round(4)
)

print(
    "\nPrimary biological programs:"
)

display(
    program_robustness[
        program_robustness[
            "Program_Class"
        ] == "Primary"
    ][
        [
            "Program",
            "HCC09",
            "HCC10",
            "Same_direction"
        ]
    ].round(4)
)

print(
    "\nClaim boundary:"
)

print(
    "Pseudotemporal compatibility "
    "does not prove true lineage conversion."
)

print(
    "\nSTEP27 FROZEN — "
    "READY FOR STEP28 FEASIBILITY ASSESSMENT"
)